In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

from torch.nn.utils.fusion import fuse_conv_bn_eval

from torch.utils.data import DataLoader, TensorDataset

import onnx

import numpy as np

from datetime import datetime
from time import time

import matplotlib.pyplot as plt

# Enable 64-bit by default for torch
torch.set_default_dtype(torch.float64)

## Evaluating HELOC

In [2]:
heloc_raw = np.genfromtxt(
    "../datasets/heloc_dataset.csv",
    delimiter=",",
    names=True,
    dtype=None,
    encoding="utf-8",
)

labels_str = heloc_raw["RiskPerformance"]
label_map = {"Bad": 0, "Good": 1}
y = np.array([label_map[value] for value in labels_str], dtype=np.int64)

feature_names = [name for name in heloc_raw.dtype.names if name != "RiskPerformance"]
X = np.column_stack([heloc_raw[name].astype(np.float64) for name in feature_names])

X_tensor = torch.from_numpy(X).to(torch.get_default_dtype())
X_tensor = X_tensor[5001:,:]
y_tensor = torch.from_numpy(y).long()
y_tensor = y_tensor[5001:]

heloc_ds = TensorDataset(X_tensor, y_tensor)

heloc_loader = DataLoader(
    heloc_ds,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# Backward compatibility for downstream cells using cifar_loader
cifar_loader = heloc_loader

print(f"HELOC samples:       {len(heloc_ds)}")
print(f"Feature dimension:   {X_tensor.shape[1]}")
print(f"Class distribution (0=Bad, 1=Good): {torch.bincount(y_tensor)}")

x_batch, y_batch = next(iter(heloc_loader))
print(f"First batch X shape: {tuple(x_batch.shape)}")
print(f"First batch y shape: {tuple(y_batch.shape)}")

HELOC samples:       5458
Feature dimension:   23
Class distribution (0=Bad, 1=Good): tensor([2784, 2674])


/home/samuel/Dokumente/Projects/NNV/FHE/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


First batch X shape: (128, 23)
First batch y shape: (128,)


In [3]:
import json

feature_stats = {
    name: {
        "min": float(max(0.0, np.min(X[:, i]))),
        "max": float(np.max(X[:, i])),
        "q1": float(np.quantile(X[:,i], 0.25)),
        "q3": float(np.quantile(X[:,i], 0.75)),
        "median": float(np.median(X[:, i])),
    }
    for i, name in enumerate(feature_names)
}

feature_stats

{'ExternalRiskEstimate': {'min': 0.0,
  'max': 94.0,
  'q1': 63.0,
  'q3': 79.0,
  'median': 71.0},
 'MSinceOldestTradeOpen': {'min': 0.0,
  'max': 803.0,
  'q1': 118.0,
  'q3': 249.5,
  'median': 178.0},
 'MSinceMostRecentTradeOpen': {'min': 0.0,
  'max': 383.0,
  'q1': 3.0,
  'q3': 11.0,
  'median': 5.0},
 'AverageMInFile': {'min': 0.0,
  'max': 383.0,
  'q1': 52.0,
  'q3': 95.0,
  'median': 74.0},
 'NumSatisfactoryTrades': {'min': 0.0,
  'max': 79.0,
  'q1': 12.0,
  'q3': 27.0,
  'median': 19.0},
 'NumTrades60Ever2DerogPubRec': {'min': 0.0,
  'max': 19.0,
  'q1': 0.0,
  'q3': 1.0,
  'median': 0.0},
 'NumTrades90Ever2DerogPubRec': {'min': 0.0,
  'max': 19.0,
  'q1': 0.0,
  'q3': 0.0,
  'median': 0.0},
 'PercentTradesNeverDelq': {'min': 0.0,
  'max': 100.0,
  'q1': 87.0,
  'q3': 100.0,
  'median': 96.0},
 'MSinceMostRecentDelq': {'min': 0.0,
  'max': 83.0,
  'q1': -7.0,
  'q3': 14.0,
  'median': -7.0},
 'MaxDelq2PublicRecLast12M': {'min': 0.0,
  'max': 9.0,
  'q1': 4.0,
  'q3': 7.0,
 

In [4]:
from onnx2pytorch import ConvertModel

import json_to_pytorch

import attack

In [5]:
data_min = [-9]*23
# different upper bounds than for Zonopoly??? But just took max of data in f_heloc.
data_max = [93, 803, 383, 383, 79, 19, 19, 100, 83, 9, 8, 104, 19, 100, 24, 66, 66, 232, 471, 32, 23, 18, 100]
class InputNorm(nn.Module):
    def __init__(self, data_min, data_max):
        super().__init__()
        data_min_t = torch.as_tensor(data_min, dtype=torch.get_default_dtype())
        data_max_t = torch.as_tensor(data_max, dtype=torch.get_default_dtype())

        self.register_buffer("data_min", data_min_t)
        self.register_buffer("scale", data_max_t - data_min_t)

    def forward(self, X):
        return (X - self.data_min) / self.scale


input_norm = InputNorm(data_min, data_max)

## Provable Bounds NN

In [60]:
# Import onnx model
onnx_model = onnx.load("../networks/heloc/heloc_2e5_gelu.onnx")
onnx.checker.check_model(onnx_model)
# convert it to pytorch
reference_model = ConvertModel(onnx_model).double()
reference_model = torch.nn.Sequential(input_norm, reference_model)

In [61]:
reference_model

Sequential(
  (0): InputNorm()
  (1): ConvertModel(
    (Gemm_linear): Linear(in_features=23, out_features=64, bias=True)
    (Gelu_gelu): GELU(approximate='none')
    (Gemm_linear_1): Linear(in_features=64, out_features=32, bias=True)
    (Gelu_gelu_1): GELU(approximate='none')
    (Gemm_output): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [62]:
reference_model.eval()

total = 0
correct = 0
nan_outputs = 0

with torch.no_grad():
    for xb, yb in heloc_loader:
        logits = reference_model(xb).squeeze(-1)  # shape: [batch]
        nan_mask = torch.isnan(logits)
        nan_outputs += nan_mask.sum().item()

        #print(logits)
        preds = (logits >= 0.0).long()

        correct += (preds == yb).sum().item()
        total += yb.numel()

acc = correct / total
nan_rate = nan_outputs / total

print(f"Total samples: {total}")
print(f"Accuracy: {acc:.6f}")
print(f"NaN outputs: {nan_outputs} ({nan_rate:.6%})")

Total samples: 5458
Accuracy: 0.726090
NaN outputs: 0 (0.000000%)


In [63]:
test_model_sampled = json_to_pytorch.json_to_pytorch("../results/gelu/heloc/heloc_2e5_gelu_27.json", double_precision=True, oob_nan=True)
test_model_sampled = torch.nn.Sequential(input_norm, test_model_sampled)

In [64]:
reference_model

Sequential(
  (0): InputNorm()
  (1): ConvertModel(
    (Gemm_linear): Linear(in_features=23, out_features=64, bias=True)
    (Gelu_gelu): GELU(approximate='none')
    (Gemm_linear_1): Linear(in_features=64, out_features=32, bias=True)
    (Gelu_gelu_1): GELU(approximate='none')
    (Gemm_output): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [65]:
feature_names = list(feature_stats.keys())
frozen = ["ExternalRiskEstimate", "MSinceOldestTradeOpen", "AverageMInFile",
          "MaxDelqEver", "MaxDelq2PublicRecLast12M"]
#adv = attack.HELOCAdversary(feature_names, feature_stats, frozen_features=frozen, epsilon_frac=0.2, attack_restarts=3,attacks_iter=10)
adv = attack.HELOCAdversary(feature_names, feature_stats, frozen_features=frozen, epsilon_frac=0.15, attack_restarts=20,attacks_iter=200, range_spec="minmax", min_one_eps=False)

In [66]:
adv.loss_strategy = attack.InternalMaxLoss("1.Gelu_gelu", "reference_model", "input")

In [67]:
def noop_loss(output, target):
    return torch.zeros(1, dtype=torch.get_default_dtype())
robust_eval_results_sampled = attack.robust_eval(reference_model, test_model_sampled, cifar_loader, noop_loss, "cpu", adv)

In [68]:
adv_examples = robust_eval_results_sampled["adv_examples"]

In [69]:
robust_eval_results_sampled["adv_examples"] = None
robust_eval_results_sampled

{'normal_loss_ref': 0.0,
 'normal_loss_test': 0.0,
 'adv_loss_ref': 0.0,
 'adv_loss_test': 0.0,
 'normal_acc_ref': 0.5100769512641994,
 'normal_acc_test': 0.5100769512641994,
 'adv_acc_ref': 0.5100769512641994,
 'adv_acc_test': 0.5100769512641994,
 'normal_eq': 1.0,
 'adv_eq': 1.0,
 'normal_nans_ref': 0.0,
 'normal_nans_test': 0.0,
 'adv_nans_ref': 0.0,
 'adv_nans_test': 0.0,
 'total': 5458,
 'adv_examples': None}

## Sampling Based GeLU

In [6]:
# Import onnx model
onnx_model = onnx.load("../networks/heloc/heloc_2e5_gelu.onnx")
onnx.checker.check_model(onnx_model)
# convert it to pytorch
reference_model = ConvertModel(onnx_model).double()
reference_model = torch.nn.Sequential(input_norm, reference_model)

In [7]:
test_model_sampled = json_to_pytorch.json_to_pytorch("../results/gelu/heloc/heloc_2e5_gelu_sampling_27.json", double_precision=True, oob_nan=True)
test_model_sampled = torch.nn.Sequential(input_norm, test_model_sampled)

In [8]:
reference_model

Sequential(
  (0): InputNorm()
  (1): ConvertModel(
    (Gemm_linear): Linear(in_features=23, out_features=64, bias=True)
    (Gelu_gelu): GELU(approximate='none')
    (Gemm_linear_1): Linear(in_features=64, out_features=32, bias=True)
    (Gelu_gelu_1): GELU(approximate='none')
    (Gemm_output): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [9]:
test_model_sampled

Sequential(
  (0): InputNorm()
  (1): Sequential(
    (0): Linear(in_features=23, out_features=64, bias=True)
    (1): ChebyshevPoly()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ChebyshevPoly()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [52]:
feature_names = list(feature_stats.keys())
frozen = ["ExternalRiskEstimate", "MSinceOldestTradeOpen", "AverageMInFile",
          "MaxDelqEver", "MaxDelq2PublicRecLast12M"]
#adv = attack.HELOCAdversary(feature_names, feature_stats, frozen_features=frozen, epsilon_frac=0.2, attack_restarts=3,attacks_iter=10)
adv = attack.HELOCAdversary(feature_names, feature_stats, frozen_features=frozen, epsilon_frac=0.15, attack_restarts=20,attacks_iter=200, range_spec="minmax", min_one_eps=False)

In [53]:
adv.loss_strategy = attack.InternalMaxLoss("1.Gelu_gelu", "reference_model", "input")

In [54]:
def noop_loss(output, target):
    return torch.zeros(1, dtype=torch.get_default_dtype())
robust_eval_results_sampled = attack.robust_eval(reference_model, test_model_sampled, cifar_loader, noop_loss, "cpu", adv)

In [55]:
adv_examples = robust_eval_results_sampled["adv_examples"]

In [56]:
robust_eval_results_sampled["adv_examples"] = None
robust_eval_results_sampled

{'normal_loss_ref': 0.0,
 'normal_loss_test': 0.0,
 'adv_loss_ref': 0.0,
 'adv_loss_test': 0.0,
 'normal_acc_ref': 0.5100769512641994,
 'normal_acc_test': 0.5100769512641994,
 'adv_acc_ref': 0.5100769512641994,
 'adv_acc_test': 0.5095272993770612,
 'normal_eq': 1.0,
 'adv_eq': 0.9994503481128618,
 'normal_nans_ref': 0.0,
 'normal_nans_test': 0.0,
 'adv_nans_ref': 0.0,
 'adv_nans_test': 0.0005496518871381459,
 'total': 5458,
 'adv_examples': None}

In [57]:
# find adv_examples in adv_examples[i][0][j] which produce nan
nan_indices = []
for i in range(len(adv_examples)):
    for j in range(len(adv_examples[i][0])):
        if torch.isnan(test_model_sampled(adv_examples[i][0][j:j+1])).any():
            nan_indices.append((i, j))
nan_indices

[(20, 24), (20, 66), (29, 58)]

In [58]:
# Print adversarial examples in nan_indices vs. their original values from X_tensor
for i, j in nan_indices:
    print(f"Adversarial example at index ({i}, {j}):\n{adv_examples[i][0][j]}")
    print(f"Original value at index ({i}, {j}):\n{X_tensor[i*128 + j]}")

Adversarial example at index (20, 24):
tensor([ 40., 177.,  58.,  64.,   0.,  17.,  13.,  14.,   0.,   0.,   2.,  33.,
          6.,  68.,  -7.,  18.,  18., 140., 168.,   6.,   5.,   0., 100.])
Original value at index (20, 24):
tensor([ 40., 177.,   1.,  64.,   4.,  14.,  10.,  29.,   2.,   0.,   2.,  17.,
          3.,  53.,  -7.,   8.,   8., 106.,  97.,   1.,   2.,   1., 100.])
Adversarial example at index (20, 66):
tensor([ 49., 200.,   0.,  71.,  57.,   0.,   3., 100.,   0.,   4.,   6.,  55.,
          6.,   7.,  -7.,   0.,  11.,  93., 471.,  24.,   5.,  12.,  84.])
Original value at index (20, 66):
tensor([ 49., 200.,   2.,  71.,  45.,   0.,   0.,  98.,   4.,   4.,   6.,  48.,
          3.,  22.,  -7.,   1.,   1.,  86., 471.,  19.,   2.,   9.,  81.])
Adversarial example at index (29, 58):
tensor([ -9., 383., 383., 383.,  10.,   1.,   0.,  95.,  -7.,   6.,   8.,   6.,
          0., 100.,  -7.,  11.,  11.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.])
Original value at index (29, 58):
tensor

In [59]:
# Summarize the differences between the original and adversarial input: Label differences with the feature name and state percentage difference in terms of the feature range
for batch_id, datapoint_id in nan_indices:
    original_input = X_tensor[batch_id * 128 + datapoint_id]
    adversarial_input = adv_examples[batch_id][0][datapoint_id]

    print(f"\nAdversarial example at batch {batch_id}, datapoint {datapoint_id}:")
    for feature_idx, feature_name in enumerate(feature_names):
        original_value = original_input[feature_idx].item()
        adversarial_value = adversarial_input[feature_idx].item()
        if original_value == adversarial_value:
            continue  # Skip features that didn't change

        feature_min = feature_stats[feature_name]["min"]
        feature_max = feature_stats[feature_name]["max"]
        feature_range = feature_max - feature_min

        if feature_range > 0:
            percentage_diff = ((adversarial_value - original_value) / feature_range) * 100
            print(f"  Feature '{feature_name}': Original={original_value:.4f}, Adversarial={adversarial_value:.4f}, Percentage difference={percentage_diff:.2f}%")
        else:
            print(f"  Feature '{feature_name}': Original={original_value:.4f}, Adversarial={adversarial_value:.4f}, Range is zero, cannot compute percentage difference")


Adversarial example at batch 20, datapoint 24:
  Feature 'MSinceMostRecentTradeOpen': Original=1.0000, Adversarial=58.0000, Percentage difference=14.88%
  Feature 'NumSatisfactoryTrades': Original=4.0000, Adversarial=0.0000, Percentage difference=-5.06%
  Feature 'NumTrades60Ever2DerogPubRec': Original=14.0000, Adversarial=17.0000, Percentage difference=15.79%
  Feature 'NumTrades90Ever2DerogPubRec': Original=10.0000, Adversarial=13.0000, Percentage difference=15.79%
  Feature 'PercentTradesNeverDelq': Original=29.0000, Adversarial=14.0000, Percentage difference=-15.00%
  Feature 'MSinceMostRecentDelq': Original=2.0000, Adversarial=0.0000, Percentage difference=-2.41%
  Feature 'NumTotalTrades': Original=17.0000, Adversarial=33.0000, Percentage difference=15.38%
  Feature 'NumTradesOpeninLast12M': Original=3.0000, Adversarial=6.0000, Percentage difference=15.79%
  Feature 'PercentInstallTrades': Original=53.0000, Adversarial=68.0000, Percentage difference=15.00%
  Feature 'NumInqLast6